# AHAAR Food Detector — Run All

Fine-tunes **YOLOv11m** on the full pan-India AHAAR dataset (218 classes: 163 dishes + 36 produce + 19 spices).
Fetches every source (~18 GB), deduplicates via `datasets/processed/class_manifest.json`, filters non-Indian pollutants, trains, exports ONNX/TFLite, and uploads to Hugging Face.

**Before Run All:** `Runtime → Change runtime type → T4 GPU`, then add two Colab secrets (🔑 icon in left sidebar):
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — from https://kaggle.com/settings
- `HF_TOKEN` (optional, only for step 7)

## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.* huggingface_hub==0.26.* kaggle six onnx onnxruntime pyyaml 2>&1 | tail -3
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1) if torch.cuda.is_available() else 'no GPU')

## 2. Config

In [ ]:
REPO_URL    = 'https://github.com/workspace220106/Aahar-Kitchen-System.git'
REPO_BRANCH = 'main'

MODEL     = 'yolo11m.pt'   # m = ~20M params, fits T4; drop to yolo11s.pt if OOM
IMG_SIZE  = 640
EPOCHS    = 80
BATCH     = 16
VAL_SPLIT = 0.10

HF_MODEL_REPO = 'your-username/aahar-detector'   # change for step 7

## 3. Clone repo + set up Kaggle

In [ ]:
import os, json, pathlib, subprocess
from pathlib import Path
from google.colab import userdata

os.chdir('/content')
if not Path('/content/aahar').exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,'/content/aahar'], check=True)

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
Path('/root/.kaggle').mkdir(exist_ok=True)
Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key':      os.environ['KAGGLE_KEY'],
}))
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Ready.')

## 4. Fetch all 5 image datasets from Kaggle (~18 GB)

In [ ]:
DATASETS = [
    ('iamsouravbanerjee/indian-food-images-dataset',                      'dish_images/souravbanerjee'),
    ('jvageesh11/foodies-ai-food-image-classification-challenge',         'dish_images/foodies'),
    ('kashyap077/indian-food-images-for-model-fine-tuning-2026',          'dish_images/kashyap'),
    ('kritikseth/fruit-and-vegetable-image-recognition',                  'ingredient_images/fruit-veg'),
    ('tankitrathaur/indian-spices-images-data',                           'ingredient_images/spices'),
]
RAW = Path('/content/raw'); RAW.mkdir(exist_ok=True)
for ref, sub in DATASETS:
    dst = RAW / sub
    if dst.exists() and any(dst.iterdir()):
        print(f'SKIP (exists): {ref}')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    print(f'--- {ref} ---')
    subprocess.run(['kaggle','datasets','download','-d',ref,'-p',str(dst),'--unzip'], check=True)
subprocess.run(['du','-sh',str(RAW)], check=True)

## 5. Merge + dedupe → YOLO layout using class_manifest.json

- Normalises class names across 5 sources (`toor dal` == `arhar dal`, `pani puri` == `paani puri`, etc.)
- Drops 43 non-Indian pollutants (pizza, burger, pasta, etc.)
- Keeps 163 Indian dishes + 36 produce + 19 spices = **218 classes**
- Converts classification folders → YOLO full-frame bounding boxes

In [ ]:
import shutil, random, re, yaml
from PIL import Image
from collections import defaultdict

manifest = json.load(open('/content/aahar/datasets/processed/class_manifest.json'))
KEEP = set(manifest['indian_dishes']) | set(manifest['produce_ingredients']) | set(manifest['spices'])
POLLUTANTS = set(manifest['excluded_non_indian'])

def canon(s):
    s = s.strip().lower().replace('-','').replace('_',' ')
    s = re.sub(r'\s+',' ', s).replace('  ',' ')
    aliases = {
     'paani puri':'pani puri','pakode':'pakoda','chapathi':'chapati','chole bature':'chole bhature',
     'dahl':'dal','rolls':'kathi roll','kaathi rolls':'kathi roll','misi roti':'missi roti',
     'misti doi':'mishti doi','malpua':'malapua','carrots':'carrot','naan bread':'naan',
     'chicken rezala':'chicken razala','khubani ka meetha':'qubani ka meetha',
     'steamed momo':'momos','sweet potatoes':'sweet potato',
    }
    return aliases.get(s, s)

SOURCES = [
    ('/content/raw/dish_images/souravbanerjee/Indian Food Images/Indian Food Images', 2),
    ('/content/raw/dish_images/foodies/Foodies_Challenge', 1),
    ('/content/raw/dish_images/kashyap/Images', 1),
    ('/content/raw/ingredient_images/fruit-veg/train', 1),
    ('/content/raw/ingredient_images/fruit-veg/validation', 1),
    ('/content/raw/ingredient_images/fruit-veg/test', 1),
    ('/content/raw/ingredient_images/spices/indian-spices-image-data', 2),
]

by_class = defaultdict(list)
for root, levels in SOURCES:
    root = Path(root)
    if not root.exists(): continue
    for class_dir in root.iterdir():
        if not class_dir.is_dir(): continue
        name = canon(class_dir.name)
        if name in POLLUTANTS or name not in KEEP: continue
        imgs = [p for p in class_dir.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
        by_class[name].extend(imgs)

class_names = sorted(by_class.keys())
cls_id = {n:i for i,n in enumerate(class_names)}
print(f'KEPT {len(class_names)} classes, {sum(len(v) for v in by_class.values())} images total')

DATA = Path('/content/aahar/data')
for sub in ('images/train','images/val','labels/train','labels/val'):
    d = DATA / sub
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

random.seed(42)
for name in class_names:
    imgs = by_class[name]; random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * VAL_SPLIT))
    for i, img in enumerate(imgs):
        split = 'val' if i < n_val else 'train'
        try:
            with Image.open(img) as im: im.verify()
        except Exception: continue
        new = f'{cls_id[name]:03d}_{img.stem[:40]}_{i:05d}{img.suffix.lower()}'
        shutil.copy2(img, DATA / f'images/{split}' / new)
        (DATA / f'labels/{split}' / new.rsplit('.',1)[0]).with_suffix('.txt').write_text(f'{cls_id[name]} 0.5 0.5 1.0 1.0\n')

cfg_path = Path('/content/aahar/aahar.yaml')
cfg_path.write_text(yaml.safe_dump({
    'path':  str(DATA),
    'train': 'images/train', 'val': 'images/val',
    'nc':    len(class_names),
    'names': class_names,
}, sort_keys=False))
print(f'train imgs: {len(list((DATA/"images/train").iterdir()))}')
print(f'val   imgs: {len(list((DATA/"images/val").iterdir()))}')

## 6. Train

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=str(cfg_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
    device=0, project='/content/runs', name='aahar_v1',
    patience=15, cache='ram', amp=True,
)
best = Path('/content/runs/aahar_v1/weights/best.pt')
print('Best:', best, best.exists())

## 7. Export ONNX + TFLite

In [ ]:
trained = YOLO(str(best))
onnx_path   = trained.export(format='onnx',  imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True)
tflite_path = trained.export(format='tflite', imgsz=IMG_SIZE, int8=False)
print('ONNX :', onnx_path)
print('TFLite:', tflite_path)

## 8. Push to Hugging Face (optional)

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN in Colab secrets first.'
api = HfApi(token=token)
create_repo(HF_MODEL_REPO, token=token, exist_ok=True, repo_type='model')
for p in [best, Path(onnx_path), Path(tflite_path)]:
    api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name, repo_id=HF_MODEL_REPO, token=token)
print(f'Uploaded to https://huggingface.co/{HF_MODEL_REPO}')